In [9]:
import sys, ollama
from pathlib import Path
project_root = Path.cwd().parent.parent.parent  # Go up from notebooks/ -> test/ -> src/ -> project root
sys.path.insert(0, str(project_root))

from config.constants import supabase, MODEL_NAME
from src.retrieval import get_document_list, get_document_toc, get_section_content
from src.assessment import generate_questions
from src.evaluation import evaluate_answer
from src.models import check_ollama, generate

print("✅ Imports successful!")
print(f"📦 Model: {MODEL_NAME}\n")

test_prompt = "What is machine learning? Answer in one sentence:"

print(f"📝 Prompt: {test_prompt}\n")
print("⏳ Generating...\n")

response = generate(test_prompt, max_tokens=100)

print(f"🤖 Response:\n{response}")

✅ Imports successful!
📦 Model: mistral

📝 Prompt: What is machine learning? Answer in one sentence:

⏳ Generating...

🤖 Response:
 Machine learning is a subset of artificial intelligence that enables computer systems to automatically learn and improve from experience without being explicitly programmed.


In [15]:
# ============================================
# CELL 3: Get Document and ToC - FIXED
# ============================================

from src.retrieval import get_document_list, get_document_toc

# Get documents
docs = get_document_list()

if not docs:
    print("❌ No documents found!")
else:
    print(f"✅ Found {len(docs)} documents:\n")
    
    # Show all documents
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc['title']}")
        print(f"   ID: {doc['id']}")
        print(f"   Sections: {doc['total_sections']}")
        print(f"   Chunks: {doc['total_chunks']}")
        print()
    
    # Find first document WITH chunks
    doc = None
    for d in docs:
        if d['total_chunks'] and d['total_chunks'] > 0:
            doc = d
            break
    
    if not doc:
        print("❌ No documents with chunks found! Run ingestion first.")
        doc_id = None
    else:
        doc_id = doc['id']
        print(f"📚 Using: {doc['title']}")
        print(f"   Sections: {doc['total_sections']}")
        print(f"   Chunks: {doc['total_chunks']}\n")
        
        # Get ToC
        toc = get_document_toc(doc_id)
        
        if not toc:
            print("❌ No ToC found!")
        else:
            print(f"✅ Found {len(toc)} top-level sections\n")
            
            # Show first 5 sections
            print("📚 Table of Contents (first 5):")
            for i, section in enumerate(toc[:5], 1):
                children_count = len(section.get('children', []))
                print(f"{i}. {section['title']} (H{section['level']}) - {children_count} subsections")

✅ Found 2 documents:

1. Title Page
   ID: d321a643-98a0-440f-82c5-52dc51bd78aa
   Sections: 67
   Chunks: None

2. Cover
   ID: 0e659d14-73b7-4095-88f6-133fcfd02b11
   Sections: 182
   Chunks: 2417

📚 Using: Cover
   Sections: 182
   Chunks: 2417

✅ Found 18 top-level sections

📚 Table of Contents (first 5):
1. Cover (H1) - 0 subsections
2. Copyright (H1) - 0 subsections
3. Table of Contents (H1) - 0 subsections
4. Preface (H1) - 9 subsections
5. Chapter 1. Introduction to Building AI Applications with Foundation Models (H1) - 5 subsections


In [5]:
# ============================================
# DEBUG: Test build_tree() directly
# ============================================

from config.constants import supabase

document_id = "0e659d14-73b7-4095-88f6-133fcfd02b11"

# Get raw nodes
raw_result = supabase.table("toc_nodes") \
    .select("id, node_id, title, level, page_start, page_end") \
    .eq("document_id", document_id) \
    .order("page_start") \
    .execute()

print(f"Raw nodes fetched: {len(raw_result.data)}")
print("\nFirst 5 nodes:")
for node in raw_result.data[:5]:
    print(f"  {node['title']} - Level {node['level']} - Pages {node['page_start']}-{node['page_end']}")

# Now test build_tree
from src.retrieval import build_tree

tree = build_tree(raw_result.data)

print(f"\n{'='*80}")
print(f"After build_tree(): {len(tree)} root nodes")
print(f"{'='*80}")

if tree:
    print("\n✅ build_tree() works!")
    print(f"\nFirst root node: {tree[0]['title']}")
    print(f"  Children: {len(tree[0].get('children', []))}")
    
    if tree[0].get('children'):
        print(f"\n  First child: {tree[0]['children'][0]['title']}")
else:
    print("\n❌ build_tree() returned empty list!")
    print("\nDebugging build_tree logic...")
    
    # Manual debug
    nodes = sorted(raw_result.data, key=lambda x: (x['page_start'], x['level']))
    print(f"\nAfter sorting: {len(nodes)} nodes")
    print(f"First node level: {nodes[0]['level']}")
    print(f"Levels present: {set(n['level'] for n in nodes[:10])}")

Raw nodes fetched: 182

First 5 nodes:
  Cover - Level 1 - Pages 1-5
  Copyright - Level 1 - Pages 6-6
  Table of Contents - Level 1 - Pages 7-12
  Preface - Level 1 - Pages 13-24
  What This Book Is About - Level 2 - Pages 14-15

After build_tree(): 18 root nodes

✅ build_tree() works!

First root node: Cover
  Children: 0


In [17]:
# ============================================
# CELL 4: Find a Good Test Section - FIXED
# ============================================

if toc:
    # Strategy: Find a section with children (actual content sections)
    test_node = None
    
    # Look for sections with subsections
    for section in toc:
        if section.get('children') and len(section['children']) > 0:
            # Use first subsection of this section
            test_node = section['children'][0]
            print(f"✅ Found good test section:")
            print(f"   Parent: {section['title']}")
            print(f"   Testing: {test_node['title']}")
            break
    
    # Fallback: use any section that's not Cover/Copyright/TOC
    if not test_node:
        for section in toc:
            if section['title'] not in ['Cover', 'Copyright', 'Table of Contents']:
                test_node = section
                print(f"✅ Using fallback section: {section['title']}")
                break
    
    # Last resort: just use first section
    if not test_node:
        test_node = toc[0]
        print(f"⚠️  Using first section: {toc[0]['title']}")
    
    print(f"\n🎯 Test section details:")
    print(f"   Title: {test_node['title']}")
    print(f"   Node ID: {test_node['node_id']}")
    print(f"   Pages: {test_node['page_start']}-{test_node['page_end']}")
else:
    test_node = None
    print("❌ No ToC available!")

✅ Found good test section:
   Parent: Preface
   Testing: What This Book Is About

🎯 Test section details:
   Title: What This Book Is About
   Node ID: h2-4-1__what-this-book-is-about
   Pages: 14-15


In [19]:
# ============================================
# CELL 5: Get Section Content
# ============================================

if test_node:
    from src.retrieval import get_section_content
    
    print("⏳ Fetching section content...\n")
    
    content = get_section_content(
        document_id=doc_id,
        node_id=test_node['node_id'],
        include_children=False
    )
    
    print(f"✅ Content retrieved:")
    print(f"   Total words: {content['total_words']:,}")
    print(f"   Total chunks: {len(content['chunks'])}")
    print(f"   Page range: {content['page_range']}")
    
    print(f"\n📖 Preview (first 300 chars):")
    print("="*80)
    print(content['text'][:300])
    print("...")
    print("="*80)

⏳ Fetching section content...

✅ Content retrieved:
   Total words: 835
   Total chunks: 6
   Page range: 14-15

📖 Preview (first 300 chars):
# Preface

##

What This Book Is About

The familiarity and ease of use of many AI engineering techniques can mislead peo‐
ple into thinking there is nothing new to AI engineering.

But while many principles
for building AI applications remain the same, the scale and improved capabilities of
AI mode
...


In [21]:
# ============================================
# CELL 6: Generate Questions - FIXED
# ============================================

if test_node and content:
    from src.assessment import generate_questions
    
    print("="*80)
    print("GENERATING QUESTIONS")
    print("="*80)
    
    print(f"\n🎯 Section: {test_node['title']}")
    print(f"⏳ Generating 3 questions... (30-60 seconds)\n")
    
    result = generate_questions(
        document_id=doc_id,
        node_id=test_node['node_id'],
        num_questions=3,
        difficulty="mixed"
    )
    
    if 'error' in result:
        print(f"❌ Error: {result['error']}")
    else:
        print(f"✅ Generated {len(result['questions'])} questions!\n")
        
        for i, q in enumerate(result['questions'], 1):
            print(f"{i}. {q.get('question', 'N/A')}")
            print(f"   Difficulty: {q.get('difficulty', 'N/A')}")
            print(f"   Pages: {q.get('page_reference', 'N/A')}\n")
        
        # Save for next cell
        test_questions = result['questions']

GENERATING QUESTIONS

🎯 Section: What This Book Is About
⏳ Generating 3 questions... (30-60 seconds)

✅ Generated 3 questions!

1. What is the primary focus of this book according to the preface?
   Difficulty: easy
   Pages: X-3

2. What does the author learn from the process of writing this book?
   Difficulty: medium
   Pages: X-3

3. What are some of the questions this book can help you answer about AI application development? (List at least three)
   Difficulty: hard
   Pages: X-3



In [23]:
# ============================================
# CELL 7: Evaluate Answer
# ============================================

if 'test_questions' in locals() and test_questions:
    from src.evaluation import evaluate_answer
    
    print("="*80)
    print("EVALUATING ANSWER")
    print("="*80)
    
    test_question = test_questions[0]['question']
    good_answer = "Machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks without being explicitly programmed for every scenario."
    
    print(f"\n📝 Question: {test_question}")
    print(f"\n💬 Answer: {good_answer}")
    print(f"\n⏳ Evaluating... (30-60 seconds)\n")
    
    evaluation = evaluate_answer(
        document_id=doc_id,
        node_id=test_node['node_id'],
        question=test_question,
        student_answer=good_answer
    )
    
    if 'error' in evaluation:
        print(f"❌ Error: {evaluation['error']}")
    else:
        print(f"📊 Score: {evaluation.get('score', 0)}/100\n")
        
        if evaluation.get('correct_points'):
            print("✅ Correct points:")
            for point in evaluation['correct_points']:
                print(f"   • {point}")
        
        if evaluation.get('missing_points'):
            print("\n⚠️  Missing points:")
            for point in evaluation['missing_points']:
                print(f"   • {point}")
        
        if evaluation.get('suggestions'):
            print(f"\n💡 Suggestions:\n   {evaluation['suggestions']}")

EVALUATING ANSWER

📝 Question: What is the primary focus of this book according to the preface?

💬 Answer: Machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks without being explicitly programmed for every scenario.

⏳ Evaluating... (30-60 seconds)

📊 Score: 90/100

✅ Correct points:
   • The primary focus of the book is to provide a framework for adapting foundation models (both large language models and large multimodal models) to specific applications.
   • The book covers various solutions and raises questions to evaluate the best solution for specific needs.

💡 Suggestions:
   Great job! You correctly identified that the book focuses on adapting foundation models to specific applications and raising questions to evaluate the best solution. However, it's important to remember that the primary focus of the book is not machine learning as mentioned in the student's answer.
